# V2a-RSN 220127_F4_run2 — Connectivity Notebook: c-GC

Prerequisite: the dataset `shared/windows` checkpoint must exist.
Parallel lane: yes. This notebook owns only its method namespace under `outputs/analysis/<DATASET_ID>/<RECORDING_ID>/<RUN_ID>/<METHOD>/`.
Progress: long stages print checkpoint-aware timing; iterative stages also display live progress bars. Reused checkpoints are reported explicitly.


In [ ]:
import sys
from pathlib import Path

try:
    _REPOSITORY_ROOT = next(
        path
        for path in (Path.cwd(), *Path.cwd().parents)
        if (path / "notebooks" / "_shared.py").is_file()
    )
except StopIteration as error:
    raise RuntimeError(
        "Could not locate the effectome repository from the notebook working directory"
    ) from error
sys.path[:0] = [
    path
    for path in (str(_REPOSITORY_ROOT / "src"), str(_REPOSITORY_ROOT))
    if path not in sys.path
]


In [ ]:
from omegaconf import OmegaConf

from effectome.connectivity import ConnectivityConfig
from notebooks._shared import (
    PROJECT_ROOT,
    make_run,
    print_stage_status,
    run_connectivity,
    stage_artifact_path,
)

DATASET_ID = "220127_F4_run2"
DATASET_OUTPUT_ID = "v2a-rsns"
RECORDING_ID = "220127_F4_run2"
RUN_ID = "reference"
METHOD = "cgc"
OUTPUT_ROOT = PROJECT_ROOT / "outputs" / "analysis" / DATASET_OUTPUT_ID / RECORDING_ID
WINDOWS_PATH = OUTPUT_ROOT / RUN_ID / "shared" / "stages" / "windows" / "artifact.pkl"
CONNECTIVITY_CONFIG_PATH = PROJECT_ROOT / "conf" / "connectivity" / "cgc.yaml"
FORCE = False
CHUNK_SIZE = 16

CONNECTIVITY_YAML = OmegaConf.to_container(
    OmegaConf.load(CONNECTIVITY_CONFIG_PATH), resolve=True
)
if not isinstance(CONNECTIVITY_YAML, dict):
    raise TypeError(f"Expected a mapping in {CONNECTIVITY_CONFIG_PATH}")
CONNECTIVITY_CFG = ConnectivityConfig(**CONNECTIVITY_YAML)

run = make_run(OUTPUT_ROOT, RUN_ID, METHOD)
print_stage_status(run)


In [ ]:
connectivity = run_connectivity(
    run,
    windows_path=WINDOWS_PATH,
    connectivity_cfg=CONNECTIVITY_CFG,
    chunk_size=CHUNK_SIZE,
    force=FORCE,
)
print(connectivity.matrices.shape)
print(connectivity.diagnostics["runtime"])
print(stage_artifact_path(run, "connectivity"))
print_stage_status(run)
